# Config

In [1]:
import os
import sys
from pathlib import Path

from fjsspw_solver.genetic_algorithm import MethodParams

# sys.path.append(str(Path(__file__).parent.parent / 'src'))

from pathlib import Path

from fjsspw_solver import GeneticAlgorithm, Individual, Encoding, GAParams
from fjsspw_solver.plotting import plot_fjsspw_gantt, plot_learning_progress, PlottingParams,plot_multiple_instance_runs
from fjsspw_solver.utils import get_next_filename

from util.benchmark_parser import WorkerBenchmarkParser

from constants import *

from pprint import pprint
from time import perf_counter

from tqdm import tqdm

In [2]:
parser = WorkerBenchmarkParser()
instance_name = 'Fattahi1'
instance_path = INSTANCE_FJSSPW_PATH / f'{instance_name}.fjs'
encoding = parser.parse_benchmark(str(instance_path))

encoding = Encoding(encoding.durations().tolist(), encoding.job_sequence())

print("-" * 20)
print("Encoding info")
print("n_operations:", encoding.n_operations())
print("n_machines:", encoding.n_machines())
print("n_workers:", encoding.n_workers())
print("n_jobs:", encoding.n_jobs())
print("-" * 20)

print(encoding.get_job_sequence())

--------------------
Encoding info
n_operations: 4
n_machines: 2
n_workers: 3
n_jobs: 2
--------------------
[0, 0, 1, 1]


In [3]:
def run_ga(params: GAParams, prints: bool = True):
    ga = GeneticAlgorithm(params, encoding, True)
    
    start = perf_counter()
    ga.run_optimization()
    end = perf_counter()

    best = ga.get_all_time_best_indv()
    if prints:
        print(f"Optimization took {end-start:.4f} seconds")
        print(f"Best fitness = {best}")
    return best

In [ ]:
def experiment(params: GAParams, N, zoom: bool = True):
    plotting_params = PlottingParams(
        instance_name = instance_name,
        bounds_file = "../../instances/InstanceData/FJSSP-W/best_known.csv",
        selection_type = "Tournament",
        crossover_type = "MOX",
        mutation_type = "Adaptive",
        selection_size = params.selection_size,
        crossover_prob = params.crossover_prob,
        mutation_prob = params.mutation_prob,
        generations = params.generations,
        population_size = params.population_size,
    )
    
    runs = []
    best_run = (None, None, None)

    log_dir = params.log_directory
    log_base_name = params.log_name
    log_output_name = params.log_output_name

    for i in tqdm(range(N)):
        log_file_i = get_next_filename(log_base_name, folder=log_dir, extension="csv")
        log_output_file_i = get_next_filename(log_output_name, folder=log_dir, extension="out")
        params.log_directory = log_dir
        params.log_name = log_file_i.name.__str__()
        params.log_output_name = log_output_file_i.name.__str__()
        best = run_ga(params, prints=False)
        runs.append((best, None, log_file_i))
        best_run = (best, log_file_i, log_output_file_i) if best_run[0] is None or best_run[0] > best else best_run

    plot_multiple_instance_runs(plotting_params, runs, zoom=zoom)

    best_fit, best_log, best_output = best_run
    if not best_output is None: 
        with open(best_output, mode='r') as f:
            fitness = float(f.readline())
            assert fitness == best_fit, "Best fit from log is not equal to output"
            sequence = [int(x) for x in f.readline().split(',')]
            machines = [int(x) for x in f.readline().split(',')]
            workers = [int(x) for x in f.readline().split(',')]
            start_times = [float(x) for x in f.readline().split(',')]
            end_times = [float(x) for x in f.readline().split(',')]
        plot_fjsspw_gantt(sequence, machines, workers, start_times, end_times, encoding, instance_name, fitness)

: 

# Experiments

In [ ]:
log_dir = "logs"
log_base_name = "genetic_adaptive_from_python"
log_output_name = "best_indv"
generations = 80000
population_size = 10
selection_size = 5
crossover_prob = 0.05
mutation_prob = 0.03
remove_clones = True
use_mutation_neighbourhoods = True

N = 1

os.makedirs(log_dir, exist_ok=True)

params = GAParams(
    log_dir,
    log_base_name,
    log_output_name,
    # generations,
    5,
    # population_size,
    2,
    selection_size,
    crossover_prob,
    mutation_prob,
    remove_clones,
    use_mutation_neighbourhoods,
)

experiment(params, N=N, zoom=False)

  0%|          | 0/1 [00:00<?, ?it/s]